# carGO PH — Blockchain Integration
**Course:** MO-IT148 — Application Development and Emerging Technologies  
**Group:** NodeBlk

**Week:** 4-5 — Blockchain Ledger Submission  
**Description:** End-to-end Web3.py pipeline that connects to a local Ganache instance, loads a compiled  
Solidity contract, and bulk-writes IoT sensor records (GPS, Temperature, RFID) from CSV files onto the  
blockchain via type-specific helper functions.

> ⚠️ **Before running:** Start Ganache → Deploy contract in Remix → paste new address in Section 2.
>
> **Week 9 edited version**

## Section 1 — Ganache Connection

In [1]:
from web3 import Web3
import pandas as pd
import time

ganache_url = "http://127.0.0.1:7545"
web3        = Web3(Web3.HTTPProvider(ganache_url))

if not web3.is_connected():
    raise ConnectionError("Cannot connect to Ganache. Ensure Ganache is running on port 7545.")

print("Connected to Ganache:", web3.is_connected())
print("Latest block number:", web3.eth.block_number)


Connected to Ganache: True
Latest block number: 1


## Section 2 — Contract Load
> ⚠️ **Update `contract_config["address"]` after every Remix redeployment.** Ganache resets on restart — old address is dead.

In [2]:
import json

with open("../contracts/IoTDataStorage_compData.json") as f:
    artifact = json.load(f)

contract_config = {
    "address": "0x8bA344300911947b128474f05Fb1d95c3120d152",  # ← PASTE NEW ADDRESS HERE after each redeploy
    "abi":     artifact["abi"]
}

iot_contract = web3.eth.contract(
    address=web3.to_checksum_address(contract_config["address"]),
    abi=contract_config["abi"]
)
web3.eth.default_account = web3.eth.accounts[0]

print("Contract loaded at:", contract_config["address"])
print("Default account:",    web3.eth.default_account)
print("Records before write:", iot_contract.functions.iotRecordCount().call())


Contract loaded at: 0x8bA344300911947b128474f05Fb1d95c3120d152
Default account: 0x1630Ad55eC06Fd3fAB2E86703D73aC2B133DfA3C
Records before write: 0


## Section 3 — Dummy Test Transaction

In [3]:
reg_tx_hash = iot_contract.functions.registerShipment(
    "TEST-001", 0, "Manila", "Cebu", 1, "VH-TEST", "DR-TEST"
).transact()
web3.eth.wait_for_transaction_receipt(reg_tx_hash)
print("Test shipment registered. TX:", reg_tx_hash.hex())

tx_hash = iot_contract.functions.storeData(
    "READ-TEST-001", "TEST-001", "DEV-TEST", "GPS", "latitude", "14.5995"
).transact()
receipt = web3.eth.wait_for_transaction_receipt(tx_hash)
print("Dummy record stored. TX:", tx_hash.hex())
print("Transaction status:", receipt.status)

dummy_record = iot_contract.functions.iotRecords(0).call()
print("Record at index 0:", dummy_record)


Test shipment registered. TX: 6c5f12dc533c25211d8df31373b0f4909d243620d70b04f2faa724a9c2479d58
Dummy record stored. TX: 6d80086cb94fa6e780d71d52019fdec9283cfe516b8ceb569e35bcf79f65b3fd
Transaction status: 1
Record at index 0: [1782307116, 'READ-TEST-001', 'TEST-001', 'DEV-TEST', 'GPS', 'latitude', '14.5995', '0x1630Ad55eC06Fd3fAB2E86703D73aC2B133DfA3C']


## Section 3b — Type-Specific Record Helpers

In [4]:
def store_gps_record(rfid_tag, device_id, data_value):
    lat, lng = data_value.split(",")
    tx = iot_contract.functions.storeGPS(
        str(rfid_tag), str(device_id), lat.strip(), lng.strip()
    ).transact()
    return web3.eth.wait_for_transaction_receipt(tx)

def store_temperature_record(rfid_tag, device_id, data_value):
    temp_int = int(float(data_value) * 10)
    tx = iot_contract.functions.storeTemperature(
        str(rfid_tag), str(device_id), temp_int
    ).transact()
    return web3.eth.wait_for_transaction_receipt(tx)

def store_rfid_record(rfid_tag, device_id, data_value):
    tx = iot_contract.functions.storeRFIDScan(
        str(rfid_tag), str(device_id), str(data_value)
    ).transact()
    return web3.eth.wait_for_transaction_receipt(tx)

def store_generic_record(reading_id, rfid_tag, device_id, data_type, data_value):
    tx = iot_contract.functions.storeData(
        str(reading_id), str(rfid_tag), str(device_id),
        str(data_type), str(data_type), str(data_value)
    ).transact()
    return web3.eth.wait_for_transaction_receipt(tx)

print("Helper functions ready.")


Helper functions ready.


## Section 3c — CSV Data Preview

In [5]:
preview_df = pd.read_csv("../data/5-iot_data_v2.csv")
print(f"Total records to write: {len(preview_df)}")
print(preview_df["data_type"].value_counts().to_dict())
print(preview_df.head(3).to_string())


Total records to write: 866
{'GPS': 400, 'Temperature': 240, 'RFID': 226}
     reading_id  rfid_tag device_id data_type            data_value            timestamp    gps_lat     gps_lng  temperature_c
0  RDG-GPS-0153  RFID-020    GPS255       GPS  14.667702,120.978645  2026-05-03 07:00:00  14.667702  120.978645            NaN
1  RDG-RFD-0082  RFID-020    RFD479      RFID              VERIFIED  2026-05-03 08:00:00        NaN         NaN            NaN
2  RDG-GPS-0001  RFID-001    GPS926       GPS  14.644554,121.026468  2026-05-03 09:00:00  14.644554  121.026468            NaN


## Section 4 — CSV Load + Bulk Write
> ⏱️ This takes ~11 minutes. Do not interrupt once started.

In [6]:
CATEGORY_MAP = {
    "Deep Freeze": 0, "Frozen": 1, "Chill/Refrigerated": 2,
    "Pharma": 3, "Cool-Chain": 4, "Dry Goods": 5,
    "Electronics": 6, "Clothing": 7, "Industrial": 8
}

def run_bulk_write():
    ship_df = pd.read_csv("../data/1-shipment_registry_v2.csv")
    print(f"Registering {len(ship_df)} shipments...")
    for _, ship in ship_df.iterrows():
        reg_hash = iot_contract.functions.registerShipment(
            ship["rfid_tag"],
            CATEGORY_MAP[ship["goods_category"]],
            ship["origin"], ship["destination"],
            int(ship["package_count"]),
            ship["vehicle_id"], ship["driver_id"]
        ).transact()
        web3.eth.wait_for_transaction_receipt(reg_hash)
        time.sleep(0.5)
    print("All shipments registered.\n")

    iot_df = pd.read_csv("../data/5-iot_data_v2.csv")
    print(f"Loaded {len(iot_df)} IoT records. Starting bulk write...")

    for index, row in iot_df.iterrows():
        try:
            data_type  = row["data_type"]
            rfid_tag   = row["rfid_tag"]
            device_id  = row["device_id"]
            data_value = str(row["data_value"])

            if data_type == "GPS":
                receipt = store_gps_record(rfid_tag, device_id, data_value)
            elif data_type == "Temperature":
                receipt = store_temperature_record(rfid_tag, device_id, data_value)
            elif data_type == "RFID":
                receipt = store_rfid_record(rfid_tag, device_id, data_value)
            else:
                receipt = store_generic_record(row["reading_id"], rfid_tag, device_id, data_type, data_value)

            print(f"{data_type} | {rfid_tag} | {data_value} | Txn: {receipt.transactionHash.hex()}")
            time.sleep(0.5)

        except Exception as e:
            print(f"Row {index} failed: {e}")

    print("\nBulk write complete.")

run_bulk_write()


Registering 50 shipments...
All shipments registered.

Loaded 866 IoT records. Starting bulk write...
GPS | RFID-020 | 14.667702,120.978645 | Txn: b91f50b33d47cd483d11e234c9eb85d90fab55b56d371c532a31989bdf2ffff4
RFID | RFID-020 | VERIFIED | Txn: 15fe892ed2fc66af2a06a47697652878f7d95892a59f61cee2d0a4e585b91da5
GPS | RFID-001 | 14.644554,121.026468 | Txn: 8476be0ffc93966fb574dc669d2e9893484d2b698a783f4788ff59280516a6a8
GPS | RFID-020 | 14.597148,120.976441 | Txn: fc0bb49b2fb1e28718dfb387967b1446a3ae121b765fa3defa75a71312bc545b
Temperature | RFID-001 | 3.2 | Txn: d192a0effdb0cd1737c574fb8a569167afac94c66e241daad7ea2b047189d3c5
GPS | RFID-020 | 14.562975,120.955133 | Txn: b0c1481255554645a3cf60e00bcf23248516b3dd8d95d05189ab9c0ef6cef910
Temperature | RFID-007 | -18.8 | Txn: 70494a88af9fe0c28dacbc2a6abdbf0540f285d9829199c33174815264d2c61b
Temperature | RFID-001 | 9.0 | Txn: 23de7ca4c9dbd05bcf6a3df2a46d21df02e423b5314115f72d9a172c85611400
GPS | RFID-007 | 14.671551,120.940095 | Txn: 0de7b7d08

## Section 5 — Verification

In [14]:
def run_verification():
    print("=== On-Chain Record Counts ===")
    print(f"  Shipments registered : {iot_contract.functions.shipmentCount().call()}")
    print(f"  Generic IoT records  : {iot_contract.functions.iotRecordCount().call()}")
    print(f"  GPS records          : {iot_contract.functions.gpsRecordCount().call()}")
    print(f"  Temperature records  : {iot_contract.functions.tempRecordCount().call()}")
    print(f"  RFID scan records    : {iot_contract.functions.rfidRecordCount().call()}")

    # --- First Real Shipment (first registered on-chain, skips TEST-001 dummy) ---
    first_tag      = iot_contract.functions.getAllRFIDTags().call()[1]
    shipment       = iot_contract.functions.getShipment(first_tag).call()
    ship_df        = pd.read_csv("../data/1-shipment_registry_v2.csv")
    first_ship_csv = ship_df[ship_df["rfid_tag"] == first_tag].iloc[0]

    print(f"\n=== First Real Shipment ({first_tag}) ===")
    print(f"  Origin             : {shipment[2]}")
    print(f"  Destination        : {shipment[3]}")
    print(f"  Category           : {shipment[1]}")
    print(f"  Vehicle ID         : {shipment[5]}")
    print(f"  Driver ID          : {first_ship_csv['driver_id']}")
    print(f"  Package Count      : {first_ship_csv['package_count']}")
    print(f"  Scheduled Delivery : {first_ship_csv['scheduled_delivery_dt']}")
    print(f"  Actual Delivery    : {first_ship_csv['actual_delivery_dt']}")
    print(f"  Status             : {first_ship_csv['shipment_status']}")
    print(f"  Delay Reason       : {first_ship_csv['delay_reason']}")
    print(f"  Registered On-Chain: {shipment[7]}")

    # --- First Real IoT Record (true first record written on-chain, by CSV/timestamp order) ---
    iot_df          = pd.read_csv("../data/5-iot_data_v2.csv")
    first_iot_row   = iot_df.iloc[0]
    first_iot_tag   = first_iot_row["rfid_tag"]
    first_iot_type  = first_iot_row["data_type"]

    print(f"\n=== First Real IoT Record/Data On-Chain ({first_iot_tag}) ===")
    print(f"  Reading ID : {first_iot_row['reading_id']}")
    print(f"  Data Type  : {first_iot_type}")
    print(f"  Data Value : {first_iot_row['data_value']}")
    print(f"  Timestamp  : {first_iot_row['timestamp']}")

    if first_iot_type == "Temperature":
        records = iot_contract.functions.getTemperaturesByRFID(first_iot_tag).call()
        record  = records[0]
        print(f"  On-Chain Temp      : {record[3] / 10}°C")
        print(f"  On-Chain Timestamp : {record[0]}")
    elif first_iot_type == "GPS":
        records = iot_contract.functions.getLocationsByRFID(first_iot_tag).call()
        record  = records[0]
        print(f"  On-Chain Latitude  : {record[3]}")
        print(f"  On-Chain Longitude : {record[4]}")
        print(f"  On-Chain Timestamp : {record[0]}")
    elif first_iot_type == "RFID":
        records = iot_contract.functions.getRFIDScansByRFID(first_iot_tag).call()
        record  = records[0]
        print(f"  On-Chain Status    : {record[3]}")
        print(f"  On-Chain Timestamp : {record[0]}")

run_verification()

=== On-Chain Record Counts ===
  Shipments registered : 51
  Generic IoT records  : 1
  GPS records          : 400
  Temperature records  : 240
  RFID scan records    : 226

=== First Real Shipment (RFID-001) ===
  Origin             : Tatalon
  Destination        : Pasig City
  Category           : 3
  Vehicle ID         : VH-001
  Driver ID          : DR-001
  Package Count      : 9
  Scheduled Delivery : 2026-05-05 20:00:00
  Actual Delivery    : 2026-05-05 20:00:00
  Status             : Delivered
  Delay Reason       : nan
  Registered On-Chain: 1782307144

=== First Real IoT Record/Data On-Chain (RFID-020) ===
  Reading ID : RDG-GPS-0153
  Data Type  : GPS
  Data Value : 14.667702,120.978645
  Timestamp  : 2026-05-03 07:00:00
  On-Chain Latitude  : 14.667702
  On-Chain Longitude : 120.978645
  On-Chain Timestamp : 1782307184


> **Note:** "First" = first by on-chain registration order, not first row in the CSV. `5-iot_data_v2.csv` is sorted by **timestamp** across all shipments, so a tag's first reading isn't necessarily row 1 of that file.
>
> | Term | Actually means |
> |---|---|
> | First Real Shipment | First shipment registered on-chain (skips `TEST-001` dummy) |
> | First Real IoT Record | First sensor reading *for that shipment's tag*, not row 1 of the IoT CSV |